In [4]:
import sys, json
from pathlib import Path
import torch

COMP_DIR    = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
DATASET_DIR = Path("/kaggle/input/datasets/suchok/my-dataset")
WEIGHTS_DIR = Path("/kaggle/input/datasets/suchok/baseline-weights") 
WORK_DIR    = Path("/kaggle/working")

print("----------")

sys.path.insert(0, str(DATASET_DIR))
import common, importlib
importlib.reload(common)
from common import *

print("----------")
set_seeds(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

print("----------")
# 1. Load Model
model = fasterrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES + 1)
model.load_state_dict(torch.load(WEIGHTS_DIR / "fasterrcnn_baseline_last.pth", map_location="cpu"))
model.to(DEVICE)
model.eval()

print("----------1")
# 2. Build Val Index & Subset GT
val_clips = list_clips(COMP_DIR / "val")
val_index = filter_by_clips(index_split(COMP_DIR / "val"), val_clips)

# === NEW: Create a 200-image subset for fast testing ===
val_subset = val_index.sample(200, random_state=SEED).reset_index(drop=True)
gt_json_subset = WORK_DIR / "gt_subset_ignore.json"

if not gt_json_subset.exists():
    print("Building GT for subset...")
    build_coco_gt(val_subset, str(gt_json_subset), include_ignore=True)

print("----------2")
# 3. Predict and Evaluate on the SUBSET
print("Running inference on 200-image subset...")
val_preds_json_subset = WORK_DIR / "val_preds_subset.json"

metrics, _ = evaluate_full_val(
    model, val_subset, strategy="letterbox", size=1280,
    gt_json=str(gt_json_subset), score_thr=0.001, device=DEVICE,
    out_json=str(val_preds_json_subset)
)

print("\n" + "="*30)
print("FAST SUBSET METRICS (200 images):")
print("="*30)
for k, v in metrics.items():
    print(f"{k:15s}: {v:.5f}")
print("="*30)

----------
----------
----------
----------1
Building GT for subset...
----------2
Running inference on 200-image subset...


pred letterbox-1280:   0%|          | 0/200 [00:00<?, ?it/s]


FAST SUBSET METRICS (200 images):
mAP            : 0.13815
AP50           : 0.28059
AP75           : 0.12298
AP_small       : 0.06553
AP_medium      : 0.14014
AP_large       : 0.27394
